In [2]:
from langgraph.graph import StateGraph, START, END, MessagesState
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from pydantic import BaseModel, Field
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_core.messages.utils import trim_messages
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv(find_dotenv())

True

In [4]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [5]:
def count_tokens_approximately(messages):
    text = " ".join((m.content or "") for m in messages)
    if hasattr(model, "get_num_tokens"):
        return model.get_num_tokens(text)
    # Fallback: rough heuristic when no model token counter is available.
    return max(1, len(text) // 4)

In [6]:
MAX_TOKENS = 150

In [7]:
def call_model(state: MessagesState):
    
    # Trim conversation history -> last N messages that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",                      
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    response = model.invoke(messages)

    return {"messages": [response]}

In [8]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [9]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [10]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Parveen."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 8
Hi, my name is Parveen.


"Hi Parveen! It's nice to meet you. How can I help you today?"

In [11]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "I am learning LangGraph."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 33
Hi, my name is Parveen.
Hi Parveen! It's nice to meet you. How can I help you today?
I am learning LangGraph.


"That's fantastic, Parveen! LangGraph is a really powerful and interesting framework for building robust, stateful LLM applications and agents. It's a great choice for more complex, multi-step AI workflows.\n\nIt can be a bit of a learning curve with concepts like state machines, nodes, edges, and compiling graphs, but it's incredibly rewarding once it clicks.\n\nHow can I help you on your learning journey? For example:\n*   Are you just getting started and looking for an introduction?\n*   Do you have specific questions about concepts (like state, nodes, edges, tools, agents, memory)?\n*   Are you working on a particular example or project and running into issues?\n*   Or would you like some ideas for what you can build with it?\n\nLet me know where you're at or what you'd like to dive into!"

In [12]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory concepts in simple words?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 11
Can you explain short term memory concepts in simple words?


'Imagine your mind has a **small, temporary notepad or a little workbench** right in front of you.\n\n**Short-term memory (STM)** is like that. It\'s where you hold a **small amount** of information that you\'re **actively thinking about or using** at this very moment.\n\nHere are the key concepts in simple words:\n\n1.  **It\'s Your "Now" Memory:**\n    *   It\'s what you\'re consciously aware of and dealing with right now.\n    *   *Example:* When someone tells you their phone number, you hold it in your short-term memory just long enough to dial it.\n\n2.  **Very Limited Space (Small Notepad):**\n    *   You can only hold a few things in your short-term memory at once. Think of it as having only about **5 to 9 "slots"** on your mental notepad.\n    *   *Example:* Trying to remember a list of 10-12 random words is hard because it exceeds this small capacity.\n\n3.  **Very Short Lifespan (Quick Erase):**\n    *   Information in short-term memory fades away very quickly if you don\'t a

In [13]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 5
What is my name?


"As an AI, I don't have access to personal information about you, so I don't know your name."

In [14]:
for item in graph.get_state({"configurable": {"thread_id": "chat-1"}}).values['messages']:
    print(item.content)
    print('-'*120)

Hi, my name is Parveen.
------------------------------------------------------------------------------------------------------------------------
Hi Parveen! It's nice to meet you. How can I help you today?
------------------------------------------------------------------------------------------------------------------------
I am learning LangGraph.
------------------------------------------------------------------------------------------------------------------------
That's fantastic, Parveen! LangGraph is a really powerful and interesting framework for building robust, stateful LLM applications and agents. It's a great choice for more complex, multi-step AI workflows.

It can be a bit of a learning curve with concepts like state machines, nodes, edges, and compiling graphs, but it's incredibly rewarding once it clicks.

How can I help you on your learning journey? For example:
*   Are you just getting started and looking for an introduction?
*   Do you have specific questions about c